# 📑 Collateral Insurance Report
 
## 🔗 Links
- **[BRD - Collateral Insurance Report](https://cepalhellas.atlassian.net/wiki/spaces/DBTC/pages/261750785/BRD+-+CollateralInsurance)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket](https://cepalhellas.atlassian.net/browse/DBTC-1525?search_id=de667b2f-4b2b-47d4-be0b-df70795a6191)**  
  Details about the story ticket, including tasks and progress.
 
- **[UAT Findings Ticket](https://cepalhellas.atlassian.net/servicedesk/customer/portal/4/ASK-4797)**  
  A list of findings from UAT.
 
---

In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
# report_args = dbutils.widgets.text("report_args", "")
# report_name = dbutils.widgets.text("report_name", "CollateralInsurance")
# run_date = dbutils.widgets.text("run_date", "2026-02-13")

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback
"""
Accessing the Hive metastore requires specific considerations:
1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.
2. To access both Unity catalog and AWS services, an access token from Amazon is required.
3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
https://docs.databricks.com/en/security/secrets/index.html
4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.
5. After you authenticate successfully querying the Hive metastore is also working.
"""
try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared"
# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    
secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)
aws_key = aws_keys[0] if aws_keys else None
# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)
    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                          
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')
    S3_client = boto3.client('s3', region_name='eu-central-1')                          

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
    return response['Parameter']['Value']
# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

# Initialization

In [0]:
athens_tz = timezone('Europe/Athens')
now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")

try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    # run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date = spark.sql(f"SELECT MAX(BusinessDate) as BusinessDate FROM `{env_prefix}-cdh`.derivative.dexposure").collect()[0][0]

business_date = run_date

print(f"now is: {now_datetime_str}")
print(f"RunDate: {run_date}")
print(f"env_prefix: {env_prefix}")

# Top 3 Insurance

In [0]:
top3insurance = spark.sql(
f'''
select
BusinessDate,
RealEstateKey,
AgreementNumber,
InsuranceStartDate,
InsuranceEndDate,
BuildingCapital,
NetInsuranceCapital,
CompanyKey,
row_number() over (partition by BusinessDate, RealEstateKey order by InsuranceEndDate desc) as insurancerank
from `{env_prefix}-cdh`.primary.insurance
'''
)

top3insurance.createOrReplaceTempView("top3insurance")

# Insurance1

In [0]:
insurance1 = spark.sql(
f'''
select
BusinessDate,
RealEstateKey,
AgreementNumber,
InsuranceStartDate,
InsuranceEndDate,
BuildingCapital,
NetInsuranceCapital,
CompanyKey
from top3insurance
where insurancerank = 1
'''
)

insurance1.createOrReplaceTempView("insurance1")

# Total Claim

In [0]:
totalclaim = spark.sql(
f'''
select 
portfolioname, 
businessdate, 
agreementkey,
sum(outstandingtotalbalance) as OutstandingTotalBalance
from `{env_prefix}-cdh`.derivative.dexposure
where BusinessDate = '{business_date}'
group by portfolioname, businessdate, agreementkey
'''
)

totalclaim.createOrReplaceTempView("totalclaim")

# Distinct Rows

In [0]:
distinct_rows = spark.sql(
f'''
select distinct tc.portfolioname, tc.businessdate, tc.agreementkey,re.RealEstateCepalKey, OutstandingTotalBalance
from `{env_prefix}-cdh`.primary.realestatebuilding re
left join `{env_prefix}-cdh`.primary.collateral co on try_cast(co.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and co.businessdate = re.BusinessDate
and CollateralStatus='Ενεργή' 
and OwnershipPercentage > 0 
left join totalclaim tc on tc.BusinessDate = re.BusinessDate and tc.PortfolioName = co.PortfolioName and tc.AgreementKey = co.AgreementKey
where re.BusinessDate = '{business_date}'
'''
)

distinct_rows.createOrReplaceTempView("distinct_rows")

# Total Claim per Property

In [0]:
totalclaim_per_property = spark.sql(
f'''
select realestatecepalkey,portfolioname,businessdate, sum(OutstandingTotalBalance) as OutstandingTotalBalance
from distinct_rows
where 1=1
group by realestatecepalkey,portfolioname,businessdate
'''
)

totalclaim_per_property.createOrReplaceTempView("totalclaim_per_property")

# Application Status

In [0]:
applstatus = spark.sql(
f'''
select distinct
re.businessdate,
re.RealEstateKey,

-- ul.portfolioName,
-- ul.SourceSystem,

array_join(collect_set(ul.ApplicationStatus), '|') as ApplicationStatus
from `{env_prefix}-cdh`.primary.realestateinfo re 
left join `{env_prefix}-cdh`.primary.collateral co on try_cast(co.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and co.businessdate = re.BusinessDate 
left join `{env_prefix}-cdh`.primary.party p on re.ownerpartykey = p.PartyKey and p.BusinessDate = re.BusinessDate and co.SourceSystem = p.SourceSystem 
and p.PortfolioName = co.PortfolioName -- VP 29/06/2026
left join `{env_prefix}-cdh`.derivative.dunderlaw ul on ul.cdi = p.cdi and ul.businessdate = re.BusinessDate 
where co.OwnershipStatus='Active'
and ul.UnderLawType = 'N3869'
and re.BusinessDate = '{business_date}'
and ul.InLawFlag = 1
group by all
'''
)
applstatus.createOrReplaceTempView("applstatus")

# Ownership Calc

In [0]:
ownership_calc = spark.sql(
f'''
select RealEstateCepalKey , PortfolioName
      , cast( 
        (case 
          when sum(PK +(CK+E)/2) + sum(case when b.CK>b.E then (b.CK-b.E)/2 else 0 end)>1 
          then 1 
          else sum(PK +(CK+E)/2) + sum(case when b.CK>b.E then (b.CK-b.E)/2 else 0 end)
        end) as DECIMAL(19,2) 
      ) as OwnershipPercentage,
      businessdate
  from (
      select a.RealEstateCepalKey, a.PortfolioName,a.businessdate
        , sum(CASE WHEN a.OwnershipTypeCode = 'Πλήρης κυριότητα' THEN a.OwnershipPercentage/100 else 0 end ) as PK
        , sum(CASE WHEN a.OwnershipTypeCode = 'Ψιλή κυριότητα' THEN a.OwnershipPercentage/100 else 0 end ) as CK
        , sum(CASE WHEN a.OwnershipTypeCode = 'Επικαρπία' THEN a.OwnershipPercentage/100 else 0 end ) as E
      from (
       select distinct re.RealEstateCepalKey, co.PortfolioName,re.businessdate,rei.OwnershipTypeCode,rei.OwnershipPercentage,rei.OwnershipKey
      from `{env_prefix}-cdh`.primary.realestatebuilding re
      join `{env_prefix}-cdh`.primary.collateral co
          on try_cast(co.RealEstateKey as string) = try_cast(re.RealEstateKey as string)
          and co.businessdate = re.BusinessDate
      join `{env_prefix}-cdh`.primary.realestateinfo rei
          on rei.RealEstateKey = re.RealEstateKey
          and rei.BusinessDate = re.BusinessDate
      where 1=1
        AND rei.OwnershipStatus = 'Active'
        and rei.OwnershipPercentage > 0
        and re.businessdate = '{business_date}'
        ) a
      group by a.RealEstateCepalKey , a.PortfolioName,a.businessdate
  ) b
  group by RealEstateCepalKey , PortfolioName, businessdate
'''
)
ownership_calc.createOrReplaceTempView("ownership_calc")


# Insurance Type Calc

In [0]:
insurance_type_calc = spark.sql(
f'''
select 
    BusinessDate,
    RealEstateKey,
    SUM(CASE WHEN GroupInsuranceFlag = true THEN 1 ELSE 0 END) as GroupInsuranceCount,
    COUNT(GroupInsuranceFlag) as TotalInsuranceCount,
    CASE 
      WHEN SUM(CASE WHEN GroupInsuranceFlag = true THEN 1 ELSE 0 END) = COUNT(ifnull(GroupInsuranceFlag,false)) THEN 'Group'
      WHEN SUM(CASE WHEN GroupInsuranceFlag = true THEN 1 ELSE 0 END) = 0 THEN 'Individual'
      ELSE 'Group-Individual' 
    END as InsuranceTypeCorrect,
    COUNT(DISTINCT AgreementNumber) as CountofContracts
  from `{env_prefix}-cdh`.primary.insurance
  where 1=1 
  group by BusinessDate, RealEstateKey
'''
)
insurance_type_calc.createOrReplaceTempView("insurance_type_calc")

# Liquidation Value

In [0]:
liquidation_value = spark.sql(
f'''
  select  
    dam.RealEstateKey,
    dam.BusinessDate,
    dam.PortfolioName,
    sum(dam.LVNetBase) over (partition by dam.businessdate, dam.realestatekey)  as `Liquidation Value`
  from  `{env_prefix}-cdh`.derivative.danalyticsmetrics dam 
  where 1=1
  and dam.BusinessDate = '{business_date}'
'''
)
liquidation_value.createOrReplaceTempView("liquidation_value")

# a1

In [0]:
a1 = spark.sql(
f'''
select 
re.BusinessDate as BusinessDate,
co.PortfolioName as Portfolio,
re.RealEstateActive as `Property Status`,
re.RealEstateCepalKey as `PropertyID`,
re.RealEstateKey,
re.RealEstateCategory as `Property Category`,
re.RealEstateSubCategory as `Property SubCategory`,
co.SubCategory as `Collateral SubCategory`,
re.RealEstateDescription as `Property Description`,
re.RealEstateAddress1 as `Address`,
re.RealEstateArea as `Area`,
re.RealEstateCity as `City`,
re.RealEstatePostalCode as `Zip Code`,
coalesce(re.RealEstateLandArea,0) as `Land Area`,
re.RealEstateNetArea as `Net Area`,
re.RealEstateConstructionYear as `Construction Year`,
cast(re.RealEstateMarketValue as decimal(19,2)) as `Market Value`,
co.RealEstateCollateralStatus as `Collateral RE Status`, 
co.CollateralREStatusChangeDate as `Collateral RE Status Change Date`, 
co.InitialLegalOrder as `Lien`,     
CAST(lv.`Liquidation Value` as DECIMAL(19,2)) as `Liquidation Value`,  
case when ul.UnderLawType = "N3869" and ul.InLawFlag=1 then "Yes" else "No" end as `Property Owner under L3869`, 
row_number() over (partition by re.BusinessDate, co.PortfolioName,  re.RealEstateCepalKey order by 
                              case when ul.UnderLawType = "N3869" and ul.InLawFlag=1 then 1 else 2 end,
                              case when co.RealEstateCollateralStatus = "Ενεργή" then 1 else 2 end) as row_num,
case when pr.productdescription like '%ΟΜΟΛΟ%' then "Yes" else "No" end as `Bond Loans`,
case when ins1.InsuranceEndDate > re.BusinessDate then "Yes" else "No" end as `Insurance Flag`,
coalesce(dparty.lastnamegr,'') as `Latest Insurance Company`, 
cast(ins1.InsuranceStartDate as date) as `Latest Insurance Date Start`,
cast(ins1.InsuranceEndDate as date) as `Latest Insurance Date End`,
re.RealEstateInsuranceCategory as `Latest Insurance Property Category`, -- VP 2/9/2026
re.RealEstateUsageCodeType, -- VP 2/9/2026
coalesce(itc.InsuranceTypeCorrect, 
         case when in.GroupInsuranceFlag = true then 'Group' 
              when in.GroupInsuranceFlag = false then 'Individual' 
              else null end) as `Type of Insurance Contract`,  
itc.CountofContracts as `#InsuranceContracts`,  
cast(tc.OutstandingTotalBalance as decimal(19,2)) as `Total Claim`,
oc.OwnershipPercentage as `Ownership Percentage`, 
aps.ApplicationStatus as `Application Status`,
CASE WHEN aps.ApplicationStatus IN ('Cancelled', 'Expelled', 'No Data', 'Other actions', 'Paid Off', 'Rejected', 'Resigned') THEN 'No' 
     WHEN aps.ApplicationStatus IN ('Pending','Pending - Approved Not Implemented','Rejected_Pending Appeal','Rejected_Pending Appeal ') THEN 'Pending' 
     WHEN aps.ApplicationStatus IN ('Accepted', 'Accepted_Pending Appeal', 'Accepted_Pending Appeal ', 'Approved Not implemented') THEN 'Yes' ELSE 'N/A' END  aS `Application Status Mapped`,    
CONCAT(party.FirstNameGR, ' ', party.LastNameGR) as `Owner Name`,--ex.PartyName as `Owner Name`,
party.TaxRegistrationNumber as `Tax Number`,--case when ex.PartyTaxRegistrationNumber is null then ex.CDI else ex.PartyTaxRegistrationNumber end as `Tax Number`,
rei.OwnershipTypeCode as `Ownership Type`,
rei.OwnershipPercentage as `Ownership Percentage1`,

party.SourceSystem as PartySourceSystem

from `{env_prefix}-cdh`.primary.realestatebuilding re
left join `{env_prefix}-cdh`.primary.collateral co on try_cast(co.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and co.businessdate = re.BusinessDate and co.SourceSystem = re.SourceSystem
and CollateralStatus='Ενεργή' 
left join `{env_prefix}-cdh`.primary.realestateinfo rei on rei.RealEstateKey = re.RealEstateKey and rei.BusinessDate = re.BusinessDate and re.SourceSystem = rei.SourceSystem
left join `{env_prefix}-cdh`.derivative.dexposure ex on ex.AgreementKey = co.AgreementKey and ex.BusinessDate = co.BusinessDate and ex.PortfolioName = co.PortfolioName
left join 
(
     select * from `{env_prefix}-cdh`.primary.party 
    -- where SourceSystem = 'thesis'
) party
on party.PartyKey = rei.OwnerPartyKey  and party.businessdate = re.BusinessDate and party.PortfolioName = co.PortfolioName --and party.SourceSystem = re.SourceSystem and party.SourceSystem = rei.SourceSystem
left join `{env_prefix}-cdh`.derivative.dunderlaw ul on ul.cdi = party.CDI and ul.businessdate = party.BusinessDate 
left join `{env_prefix}-cdh`.primary.product pr on pr.ProductKey = ex.ProductKey and pr.BusinessDate = ul.BusinessDate
left join `{env_prefix}-cdh`.primary.insurance in on try_cast(in.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and in.BusinessDate = re.BusinessDate
left join insurance1 ins1 on try_cast(ins1.RealEstateKey as string) = try_cast(in.RealEstateKey as string) and ins1.BusinessDate = in.BusinessDate
left join totalclaim_per_property tc on re.RealEstateCepalKey = tc.realestatecepalkey and tc.PortfolioName = co.PortfolioName and tc.businessdate = re.BusinessDate
left join applstatus aps on aps.RealEstateKey = re.RealEstateKey and aps.BusinessDate = re.BusinessDate
left join liquidation_value lv on lv.RealEstateKey = re.RealEstateCepalKey and lv.BusinessDate = re.BusinessDate
and lv.PortfolioName=ex.PortfolioName 
left join `{env_prefix}-cdh`.derivative.dparty on dparty.PartyKey = ins1.companykey and dparty.BusinessDate = ins1.BusinessDate 
AND dparty.PartyType = 'Organization' AND dparty.SourceSystem = co.SourceSystem
left join ownership_calc oc on oc.RealEstateCepalKey = re.RealEstateCepalKey and  oc.PortfolioName = co.PortfolioName and oc.businessdate=co.businessdate 
left join insurance_type_calc itc on itc.RealEstateKey = re.RealEstateKey and itc.BusinessDate = re.BusinessDate  
where 1=1
and co.portfolioname is not null
and re.businessdate = '{business_date}'
--and re.SourceSystem = 'thesis'
and rei.OwnershipPercentage>0
and rei.OwnershipStatus='Active' 
--and rei.SourceSystem = 'thesis'
--and party.SourceSystem = 'thesis'
'''
)
a1.createOrReplaceTempView("a1")

# Portfolios per Property

In [0]:
portfolios_per_property = spark.sql(
f'''
  select
      d.PropertyId,
      COUNT(DISTINCT d.Portfolio) as `#Portfolios`
  from a1 d
  group by d.PropertyId
'''
)
portfolios_per_property.createOrReplaceTempView("portfolios_per_property")

# Min Lien

In [0]:
minliencte = spark.sql(
f'''
  SELECT  PropertyID, Portfolio, MIN(Lien) AS MinLien 
  FROM a1 
  GROUP BY PropertyID, Portfolio -- VP 29/06/2026
'''
)
minliencte.createOrReplaceTempView("minliencte")

# final table

In [0]:
finaltbl = spark.sql(
f'''
select DISTINCT 
a1.BusinessDate,
a1.Portfolio,
a1.`Property Status`,                
a1.PropertyId,                      
a1.`Property Category`,            
a1.`Property SubCategory`,      
a1.`Collateral SubCategory`,  
a1.`Property Description`,      
a1.`Address`,
a1.`Area`,
a1.`City`,
a1.`Zip Code`,
a1.`Land Area`,
a1.`Net Area`,
coalesce(a1.`Construction Year`, '') as `Construction Year`,
a1.`Market Value`,
a1.`Collateral RE Status`, 
a1.`Collateral RE Status Change Date`, 
ml.MinLien as `Lien`,
cast(a1.`Liquidation Value` as decimal(19,2)) as `Liquidation Value` ,                             
a1.`Property Owner under L3869`, 
a1.`Bond Loans`,
a1.`Insurance Flag`,
a1.`Latest Insurance Company`, 
a1.`Latest Insurance Date Start`, 
a1.`Latest Insurance Date End`, 
IFNULL(IFNULL(a1.`Latest Insurance Property Category`,m.InsuranceTypeDescriptionRE), 'For Manual Calculation') as `Latest Insurance Property Category`, -- VP 2/9/2026
a1.`Type of Insurance Contract`, 
ifnull(a1.`#InsuranceContracts`, 0) as CountOfInsuranceContracts,
a1.`Total Claim`, 
ppp.`#Portfolios` as CountOfPortfolios,
a1.`Ownership Percentage`, 
a1.`Application Status`, 
a1.`Application Status Mapped`,
cast(a1.`Net Area`*cc.ConstructionCConst*(1-ifnull(d1.DepreciationPercentage,d2.DepreciationPercentage)) as decimal(19,2)) as ConstructionCost,
cast((a1.`Net Area`*cc.ConstructionCConst*(1-ifnull(d1.DepreciationPercentage,d2.DepreciationPercentage)))* a1.`Ownership Percentage` as decimal(19,2)) as TotalSumInsured,
a1.`Owner Name`, 
a1.`Tax Number`,
a1.`Ownership Type`,
cast(a1.`Ownership Percentage1` as decimal(19,2)),
CASE WHEN a1.row_num = 1 THEN 'CollateralInsuranceData' ELSE NULL END AS FileFlag, 
a1.PartySourceSystem,
current_timestamp() as IngestionTimeStamp 
from a1
left join MinLienCTE ml ON a1.PropertyID = ml.PropertyID and a1.Portfolio = ml.Portfolio  -- VP 29/06/2026
left join portfolios_per_property ppp ON ppp.PropertyId = a1.PropertyID
left join `{env_prefix}-reportinputfiles`.internal.collateral_insurance_category_params m on m.UsageCodeDescr = a1.RealEstateUsageCodeType and m.CatDescr = a1.`Property SubCategory` -- VP 2/9/2026
left join `{env_prefix}-reportinputfiles`.internal.collateral_insurance_cc_params cc on lower(cc.InsuranceTypeDescription) = lower(IFNULL(a1.`Latest Insurance Property Category`,m.InsuranceTypeDescriptionRE)) -- VP 2/9/2026
left join `{env_prefix}-reportinputfiles`.internal.collatera_insurance_depriciation_params d1 on lower(d1.InsuranceTypeDescription) = lower(IFNULL(a1.`Latest Insurance Property Category`,m.InsuranceTypeDescriptionRE)) and (try_cast(a1.`Construction Year` as int) >=year(d1.startdate)and try_cast(a1.`Construction Year` as int) <=year(d1.enddate)) -- VP 2/9/2026
left join `{env_prefix}-reportinputfiles`.internal.collatera_insurance_depriciation_params d2 on lower(d2.InsuranceTypeDescription) = lower(IFNULL(a1.`Latest Insurance Property Category`,m.InsuranceTypeDescriptionRE)) and d2.AgeGroup = 'New built (2001-2020)' -- VP 2/9/2026
'''
)
finaltbl.createOrReplaceTempView("finaltbl")

In [0]:
target_table = f"`{env_prefix}-cdh-delivery`.internalreports.collateralinsurance"

spark.sql(f"""DELETE FROM {target_table} WHERE BusinessDate = '{business_date}'""")

(finaltbl.write
 .format("delta")
 #.option("overwriteSchema", "true")
 .mode("append")
 .saveAsTable(target_table))


# Pre invoicing

In [0]:
pre_invoicing = spark.sql(
f'''

select distinct
co.PortfolioName as Portfolio,
re.businessdate,
re.RealEstateCepalKey as `PropertyKey`,
re.RealEstateActive as `PropertyStatus`,
ag.AgreementKey as `AgreementNumber`,
ex.CDI as `PrimaryCDI`,
ex.AssetSegment as `Segment`,
cast(tc.OutstandingTotalBalance as decimal(19,2)) as `TotalBalance`,
--co.RealEstateCollateralStatus as CollateralREStatus, -- MC inclusion for checks - this field exists On-prem
case when max(tc.OutstandingTotalBalance) over (partition by ex.BusinessDate, re.RealEstateKey)  =  tc.OutstandingTotalBalance  then 'V' else '-' end as  `MaxTotalBalance`,
re.RealEstateDescription as `PropertyDescription`,
current_timestamp() as IngestionTimeStamp
from `{env_prefix}-cdh`.primary.realestatebuilding re
left join `{env_prefix}-cdh`.primary.collateral co on try_cast(co.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and co.businessdate = re.BusinessDate 
left join `{env_prefix}-cdh`.derivative.dexposure ex on ex.AgreementKey = co.AgreementKey and ex.BusinessDate = co.BusinessDate and ex.PortfolioName = co.PortfolioName
left join `{env_prefix}-cdh`.primary.agreement ag on ag.AgreementKey = ex.AgreementKey and ag.BusinessDate = ex.BusinessDate and ag.PortfolioName = ex.PortfolioName
left join `{env_prefix}-cdh`.primary.insurance in on try_cast(in.RealEstateKey as string) = try_cast(re.RealEstateKey as string) and in.BusinessDate = re.BusinessDate 
left join totalclaim tc on tc.BusinessDate = re.BusinessDate and tc.PortfolioName = co.PortfolioName and tc.AgreementKey = co.AgreementKey
left join `{env_prefix}-cdh`.primary.realestateinfo rei on rei.RealEstateKey = re.RealEstateKey and rei.BusinessDate = re.BusinessDate 
where 1=1
and ag.AgreementKey is not null
and re.businessdate = '{business_date}'
and co.portfolioname is not null
and co.PortfolioName <> 'Core Retail' -- MC change (we exclude Core Retail portfolio based on BRD)
order by 2
'''
)

pre_invoicing.createOrReplaceTempView("pre_invoicing")

In [0]:
pre_invoicing_target_table = f"`{env_prefix}-cdh-delivery`.internalreports.collateralinsurance_pre_invoicing"

spark.sql(f"""DELETE FROM {pre_invoicing_target_table} WHERE BusinessDate = '{business_date}'""")

(pre_invoicing.write
 .format("delta")
 #.option("overwriteSchema", "true")
 .mode("append")
 .saveAsTable(pre_invoicing_target_table))
